# Train on PanopTILs Refined

Use all prepared patches from the patient-separated dataset. Start with
[Prepare PanopTILs](prepare-data.ipynb), which creates the images, nucleus instances
and cell types used here. Tissue-area maps are never loaded.

Run from `docs/tutorials/` in the repository environment. A CUDA GPU is preferred.
This full-data run takes much longer than a few optimization steps. Five epochs are
an example starting point; use the held-out results to decide whether further
training is needed. Do not treat a lower loss as proof of segmentation quality.


## Setup

In [ ]:
import json
import random
from pathlib import Path

import albumentations as A
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from matplotlib.patches import Patch
from skimage.color import label2rgb
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader

from cellseg_models_pytorch.inference.post_processor import PostProcessor
from cellseg_models_pytorch.losses import CELoss, DiceLoss, JointLoss, MSE, MultiTaskLoss
from cellseg_models_pytorch.metrics import panoptic_quality
from cellseg_models_pytorch.models.cellpose import CellPose
from cellseg_models_pytorch.transforms.albu_transforms import (
    AlbuStrongAugment, ApplyEach, CellposeTransform, MinMaxNormalization,
)
from cellseg_models_pytorch.transforms.functional.normalization import minmax_normalize
from cellseg_models_pytorch.torch_datasets import TrainDatasetFolder
from cellseg_models_pytorch.utils import FileHandler

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.set_num_threads(2)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data_root = Path("../../data/panoptils-nuclei")
class_names = [
    "background", "neoplastic", "stromal", "inflammatory",
    "epithelial", "other", "unknown",
]
print(f"Device: {device}")


## Load aligned patches

In [ ]:
train_transforms = A.Compose([
    A.HorizontalFlip(p=0.33),
    A.VerticalFlip(p=0.33),
    AlbuStrongAugment(seed=seed),
    MinMaxNormalization(copy=True),
], seed=seed)
validation_transforms = A.Compose([MinMaxNormalization(copy=True)])
instance_transforms = ApplyEach([CellposeTransform(deduplicate=False)], as_list=True)

train_dataset = TrainDatasetFolder(
    str(data_root / "train/images"), str(data_root / "train/labels"),
    mask_keys=("inst", "type"), transforms=train_transforms,
    inst_transforms=instance_transforms,
)
validation_dataset = TrainDatasetFolder(
    str(data_root / "validation/images"), str(data_root / "validation/labels"),
    mask_keys=("inst", "type"), transforms=validation_transforms,
    inst_transforms=instance_transforms,
)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=4)
validation_loader = DataLoader(
    validation_dataset, batch_size=8, shuffle=False, num_workers=4,
)
print(f"Patches: {len(train_dataset)} training, {len(validation_dataset)} validation")


## Training recipe

This nuclei-only recipe follows the
[Histolytics training example](https://hautaniemilab.github.io/histolytics/user_guide/seg/finetuning/).
Use AdamW and a cosine learning-rate schedule. The cell-type loss combines
cross-entropy with spectral decoupling and Dice; mean squared error trains the flows.
`MultiTaskLoss` gives those two heads equal weight. The tissue head and semantic-area
loss from the panoptic example are deliberately omitted.

Training uses aligned flips and image-only colour augmentation. Flow targets are
computed from the augmented instance labels. Validation has normalization only,
with no random augmentation or dropped samples.


In [ ]:
criterion = MultiTaskLoss({
    "cellpose": JointLoss([MSE()]),
    "type_mask": JointLoss([CELoss(apply_sd=True), DiceLoss()]),
}, loss_weights={"cellpose": 1.0, "type_mask": 1.0}).to(device)

learning_rate = 1e-4
num_epochs = 5


## Initialize the network

Start with an ImageNet-pretrained EfficientNet-B5 encoder and new segmentation
heads. This is training a segmentation model, not loading pretrained segmentation
weights. The first run downloads the encoder weights.

In [ ]:
model = CellPose(
    7, enc_name="efficientnet_b5", enc_pretrain=True, device=device,
).model
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=0)
checkpoint_dir = Path("training-checkpoints")
checkpoint_dir.mkdir(exist_ok=False)


## Reuse the epoch steps

Fail on a nonfinite loss so invalid batches cannot silently change the experiment.

In [ ]:
def train_epoch(model, loader, optimizer):
    total = 0.0
    for batch in loader:
        images = batch["image"].to(device)
        targets = {
            "type_mask": batch["type"].long().to(device),
            "cellpose": batch["cellpose_inst"].float().to(device),
        }
        optimizer.zero_grad(set_to_none=True)
        output = model(images)["nuc"]
        loss = criterion({"type_mask": output.type_map, "cellpose": output.aux_map}, targets)
        if not torch.isfinite(loss):
            raise RuntimeError("Nonfinite training loss; stop and check inputs")
        loss.backward()
        optimizer.step()
        total += float(loss.detach()) * images.shape[0]
    return total / len(loader.dataset)


def validate_epoch(model, loader):
    total = 0.0
    with torch.inference_mode():
        for batch in loader:
            images = batch["image"].to(device)
            targets = {
                "type_mask": batch["type"].long().to(device),
                "cellpose": batch["cellpose_inst"].float().to(device),
            }
            output = model(images)["nuc"]
            loss = criterion({"type_mask": output.type_map, "cellpose": output.aux_map}, targets)
            if not torch.isfinite(loss):
                raise RuntimeError("Nonfinite validation loss; stop and check inputs")
            total += float(loss) * images.shape[0]
    return total / len(loader.dataset)


def save_checkpoint(epoch, phase):
    numpy_rng = np.random.get_state()
    path = checkpoint_dir / f"epoch_{epoch:02d}.pt"
    with path.open("xb") as destination:
        torch.save({
            "model": model.state_dict(), "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(), "epoch": epoch, "phase": phase,
            "encoder": model.enc_name, "class_names": class_names,
            "torch_rng": torch.get_rng_state(),
            "cuda_rng": torch.cuda.get_rng_state(device) if device.type == "cuda" else None,
            "python_rng": random.getstate(),
            "numpy_rng": (numpy_rng[0], numpy_rng[1].tolist(), *numpy_rng[2:]),
            "data_manifest": json.loads((data_root / "manifest.json").read_text()),
            "normalization": "global min-max, RGB float32",
            "augmentation_seed": seed, "num_workers": train_loader.num_workers,
            "num_epochs": num_epochs,
            "training_loss": training_loss, "validation_loss": validation_loss,
        }, destination)
    return path


## Train and validate

In [ ]:
for epoch in range(num_epochs):
    train_transforms.set_random_seed(seed + epoch)
    model.train()
    training_loss = train_epoch(model, train_loader, optimizer)
    model.eval()
    validation_loss = validate_epoch(model, validation_loader)
    scheduler.step()
    saved_checkpoint = save_checkpoint(epoch + 1, "unfrozen")
    print(
        f"Epoch {epoch + 1}: training loss {training_loss:.3f}, "
        f"validation loss {validation_loss:.3f}"
    )


## Reload and check the saved model

Each epoch writes a new checkpoint with model, optimizer, scheduler and random
states, class meanings, preprocessing and the patient split. An existing checkpoint
directory is not overwritten; choose a new name for another run. Checkpoints are
saved at epoch boundaries. Keep the augmentation seed and worker count when resuming;
stateful or persistent workers and mid-epoch resumption need their own loader state.

Reload with the same architecture and restricted loading. For an interrupted run,
continue from the stored epoch with the restored optimizer and scheduler. This
example uses float32, so no mixed-precision scaler is needed.


In [ ]:
state = torch.load(saved_checkpoint, map_location="cpu", weights_only=True)
inference_model = CellPose(
    len(state["class_names"]), enc_name=state["encoder"], enc_pretrain=False, device=device,
)
restored = inference_model.model
restored.load_state_dict(state["model"], strict=True)
resumed_optimizer = torch.optim.AdamW(restored.parameters(), lr=learning_rate)
resumed_scheduler = CosineAnnealingLR(resumed_optimizer, T_max=state["scheduler"]["T_max"])
resumed_optimizer.load_state_dict(state["optimizer"])
resumed_scheduler.load_state_dict(state["scheduler"])

random.setstate(state["python_rng"])
np.random.set_state(state["numpy_rng"])
torch.set_rng_state(state["torch_rng"])
if device.type == "cuda" and state["cuda_rng"] is not None:
    torch.cuda.set_rng_state(state["cuda_rng"], device)

inference_model.set_inference_mode(mixed_precision=False)
inference_model.post_processor = PostProcessor("cellpose", {"use_gpu": device.type == "cuda"})
print("Reloaded model, optimizer and scheduler after epoch", state["epoch"])


## Check held-out instances and cell types

Predict complete 1024-pixel regions at their original scale, then reconstruct the
instances. Evaluate 16 evenly spaced validation regions chosen before seeing results.
Panoptic quality (PQ) measures matched nuclei and their mask overlap; 1 is perfect
and 0 means no qualifying matches. The table reports mean PQ per region, plus object
counts, for all nuclei and for each cell type. A class is scored only where a
target or prediction exists; NaN means there were no such objects to evaluate. The class scores require both the
correct type and a matching instance. Inspect rare classes rather than relying on
one overall score. These examples are not a full dataset benchmark.


In [ ]:
region_paths = sorted((data_root / "validation/regions/images").glob("*.png"))
# Fix the evaluation subset before seeing predictions; do not choose the best-looking outputs.
evaluation_paths = region_paths[::max(1, len(region_paths) // 16)][:16]
records = []
for path in evaluation_paths:
    raw_image = FileHandler.read_img(path)
    labels = FileHandler.read_mat(data_root / "validation/regions/labels" / f"{path.stem}.mat")
    image = minmax_normalize(raw_image, copy=True)
    result = inference_model.post_process(
        inference_model.predict(image), use_async_postproc=False,
    )
    predicted_instances, predicted_classes = result["nuc"][0]
    records.append({
        "class": "all nuclei", "PQ": panoptic_quality(labels["inst"], predicted_instances)["pq"],
        "target objects": len(np.unique(labels["inst"][labels["inst"] > 0])),
        "predicted objects": len(np.unique(predicted_instances[predicted_instances > 0])),
    })
    for class_id, name in enumerate(class_names[1:], start=1):
        true_class = np.where(labels["type"] == class_id, labels["inst"], 0)
        predicted_class = np.where(predicted_classes == class_id, predicted_instances, 0)
        true_count = len(np.unique(true_class[true_class > 0]))
        predicted_count = len(np.unique(predicted_class[predicted_class > 0]))
        score = panoptic_quality(true_class, predicted_class)["pq"]
        records.append({
            "class": name, "PQ": score if true_count + predicted_count else np.nan,
            "target objects": true_count, "predicted objects": predicted_count,
        })
metrics = pd.DataFrame(records).groupby("class", sort=False).agg({
    "PQ": "mean", "target objects": "sum", "predicted objects": "sum",
}).round(3)
print(f"Held-out regions: {len(evaluation_paths)}")
metrics


## Compare the masks

Both rows show the same held-out region. Compare target and predicted instances
and cell types using the shared legend. Missed nuclei, merged objects and incorrect
cell types should remain visible; do not replace this view with dense class scores.

In [ ]:
# Show the first predetermined held-out region, not a selected success case.
path = evaluation_paths[0]
raw_image = FileHandler.read_img(path)
labels = FileHandler.read_mat(data_root / "validation/regions/labels" / f"{path.stem}.mat")
image = minmax_normalize(raw_image, copy=True)
result = inference_model.post_process(inference_model.predict(image), use_async_postproc=False)
predicted_instances, predicted_classes = result["nuc"][0]

# Fixed colours keep each class identical in target and prediction panels.
class_palette = np.array([
    [0, 0, 0], [0.9, 0.2, 0.2], [0.2, 0.5, 0.9], [0.2, 0.75, 0.3],
    [0.8, 0.3, 0.8], [0.95, 0.7, 0.15], [0.6, 0.6, 0.6],
])
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes[0, 0].imshow(raw_image)
axes[0, 0].set_title("Held-out image")
axes[0, 1].imshow(label2rgb(labels["inst"], bg_label=0))
axes[0, 1].set_title("Target instances")
axes[0, 2].imshow(class_palette[labels["type"].astype(int)])
axes[0, 2].set_title("Target cell types")
axes[1, 0].imshow(raw_image)
axes[1, 0].set_title("Held-out image")
axes[1, 1].imshow(label2rgb(predicted_instances, bg_label=0))
axes[1, 1].set_title("Predicted instances")
axes[1, 2].imshow(class_palette[predicted_classes])
axes[1, 2].set_title("Predicted cell types")
for axis in axes.flat:
    axis.set(xticks=[], yticks=[])
fig.legend([Patch(color=colour) for colour in class_palette[1:]], class_names[1:],
           loc="lower center", ncol=6)
fig.tight_layout(rect=(0, 0.04, 1, 1))
plt.show()
